# アルゴリズム論Ⅱ 

## 第3回：ニュートン法と実務での落とし穴（IRRの計算を題材に）

前回の「第2回：二分法での計算」では、中間値の定理を利用して確実に解を挟み込む方法を学びました。
今回は、関数の**微分（接線の傾き）**を利用して驚異的なスピードで解に収束する **ニュートン法（Newton-Raphson法）** を学びます。

ビジネスの現場で頻繁に使われる投資採算性指標 **「内部収益率（IRR: Internal Rate of Return）」** の算出を題材に：
1. ニュートン法の圧倒的な計算スピード（わずか数回で収束）
2. **「なぜExcelのIRR関数でも `#NUM!` エラーが発生するのか？」**（実務で直面する3つの致命的な落とし穴）
3. 現場におけるアルゴリズム選択（二分法とニュートン法の使い分け）
について探求します。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import warnings
import logging

# 警告・フォント検索ログの抑制
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# グラフ描画のスタイル設定
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 日本語フォント対策（自動検出）
try:
    import japanize_matplotlib
except ImportError:
    candidate_fonts = ['Noto Sans CJK JP', 'IPAexGothic', 'IPAGothic', 'Hiragino Sans', 'Yu Gothic', 'Meiryo']
    available_fonts = {f.name for f in fm.fontManager.ttflist}
    found_font = next((f for f in candidate_fonts if f in available_fonts), None)
    if found_font:
        plt.rcParams['font.family'] = found_font

plt.rcParams['axes.unicode_minus'] = False
print("✅ 環境準備完了")


---
## 1. ニュートン法の原理（接線による解の更新）

方程式 $f(x) = 0$ の解を求めるとき、現在地 $x_k$ における関数の接線を引き、その**接線が $x$ 軸と交わる点**を次の推測値 $x_{k+1}$ とする手法がニュートン法です。

点 $(x_k, f(x_k))$ における接線の方程式は：
$$ y - f(x_k) = f'(x_k) (x - x_k) $$

接線が $x$ 軸と交わる点（$y = 0$）を $x_{k+1}$ とおくと：
$$ 0 - f(x_k) = f'(x_k) (x_{k+1} - x_k) $$
これを $x_{k+1}$ について解くと、ニュートン法の更新式が得られます：

$$ x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)} $$

二分法がステップごとに誤差が半分（$1/2$）になる「一次収束」だったのに対し、ニュートン法はステップごとに**正しい有効桁数が約2倍になる「二次収束」**という圧倒的なスピードを誇ります。


---
## 2. 内部収益率（IRR）の計算式と導関数

初期投資額 $I_0$（年 $t=0$ の支出）、各年 $t$ のキャッシュフローを $C_t$ とすると、割引率（年利回り）$r$ における正味現在価値（NPV）は：
$$ \text{NPV}(r) = -I_0 + \sum_{t=1}^T \frac{C_t}{(1 + r)^t} $$

IRR とは、$\text{NPV}(r) = 0$ となる利回り $r$ のことです。

ニュートン法を適用するためには、導関数 $\text{NPV}'(r)$（$r$ に関する微分）が必要です。
$\frac{d}{dr}(1+r)^{-t} = -t(1+r)^{-(t+1)}$ ですから：

$$ \text{NPV}'(r) = -\sum_{t=1}^T \frac{t \cdot C_t}{(1 + r)^{t+1}} $$

このように、IRR の計算式は数式として綺麗に微分できるため、ニュートン法を直接適用することが可能です。


---
## 3. 成功例：素直な案件での圧倒的な計算スピード

まずは、前回扱った標準的な投資案件のキャッシュフローでニュートン法を試してみましょう。
- 0年目: $-1000$ 万円（初期投資）
- 1〜5年目: $250, 320, 380, 420, 280$ 万円（回収）


In [ ]:
cash_flows = [-1000, 250, 320, 380, 420, 280]

def npv(r):
    """NPV(r) の計算"""
    total = 0.0
    for t, c in enumerate(cash_flows):
        total += c / ((1.0 + r)**t)
    return total

def npv_prime(r):
    """NPV'(r) の計算（導関数）"""
    total = 0.0
    for t, c in enumerate(cash_flows):
        if t > 0:
            total += -t * c / ((1.0 + r)**(t + 1))
    return total

# 初期値: 年利 10% (r = 0.1) からスタート
r = 0.10

print(f"初期値: r_0 = {r * 100:.2f}%")
print("-" * 65)

# ニュートン法のループ
for step in range(1, 10):
    val = npv(r)
    slope = npv_prime(r)
    
    # ニュートン法の更新: r_{k+1} = r_k - f(r_k) / f'(r_k)
    r_next = r - (val / slope)
    diff = abs(r_next - r)
    
    print(f"Step {step}: r = {r_next * 100:.8f}%,  NPV = {val:+.4e},  変化量 = {diff:.4e}")
    r = r_next
    
    # 変化量が 1e-8 未満になったら収束とみなす
    if diff < 1e-8:
        print("-" * 65)
        print(f"🎉 わずか {step} 回の反復で完全収束しました！")
        break

print(f"求まった IRR: {r * 100:.4f}%")


前回、二分法では $10^{-6}$ の精度を出すのに **16〜20回** の反復が必要でした。
一方、ニュートン法は **わずか4回の反復** で10桁以上一致する超高精度に達しました！

**「これだけ速いなら、全部ニュートン法で解けばいいのでは？」** と思えます。
しかし、ここからが実務における大きな落とし穴です。


---
## 4. 実務での落とし穴 1：特異点 $r = -1$（年利 $-100\%$）への飛び火

NPV の式には分母に $(1 + r)$ があります。
もし $r = -1$ になると **ゼロ除算エラー** となり、さらに $r < -1$ になると小数の負の数乗や複素数が発生してプログラムが即死（クラッシュ）します。

初期値を少し離れた場所（例: $r_0 = 100\%$ や $-20\%$ など）に置いたとき、接線の勢いが強すぎて **$r < -1$ の禁止領域に吹き飛んでしまう** 現象を観察してみましょう。


In [ ]:
# 初期値をあえて年利 150% (r = 1.50) から始めてみる
r = 1.50

print(f"初期値: r_0 = {r * 100:.1f}%")
try:
    for step in range(1, 10):
        val = npv(r)
        slope = npv_prime(r)
        r_next = r - (val / slope)
        print(f"Step {step}: r = {r_next * 100:.2f}%")
        
        if r_next <= -1.0:
            print(f"💥 警告！ r が -100% 以下の禁止領域 ({r_next * 100:.2f}%) にジャンプしました！")
            print("これ以上計算を続けるとゼロ除算や複素数エラーでプログラムがクラッシュします。")
            break
        r = r_next
except Exception as e:
    print(f"エラー発生: {e}")


---
## 5. 実務での落とし穴 2 & 3：多重解と発散（Excelの `#NUM!` エラーの正体）

現実の事業では、途中で**追加投資（修繕費、リニューアル費用）**や**撤退費用（原状回復費、環境対策費）**が発生し、キャッシュフローの符号が複数回反転することがよくあります。

### 実例：油田開発や鉱山プロジェクト
- **0年目**: 初期開発投資 $-100$ 億円
- **1年目**: 採掘による大黒字 $+230$ 億円
- **2年目**: 鉱山閉鎖・環境浄化費用 $-132$ 億円

このプロジェクトの正味現在価値 $\text{NPV}(r)$ をグラフに描いてみましょう。


In [ ]:
cf_mine = [-100, 230, -132]

def npv_mine(r):
    return cf_mine[0] + cf_mine[1]/(1+r) + cf_mine[2]/((1+r)**2)

def npv_mine_prime(r):
    return -cf_mine[1]/((1+r)**2) - 2*cf_mine[2]/((1+r)**3)

# グラフの描画
r_vals = np.linspace(-0.05, 0.35, 200)
npv_vals = [npv_mine(r) for r in r_vals]

plt.figure(figsize=(9, 5))
plt.plot(r_vals * 100, npv_vals, 'b-', linewidth=2.5, label='NPV 曲線')
plt.axhline(0, color='black', linestyle='--', linewidth=1)

# 解の位置（r = 10% と r = 20%）
plt.plot(10, 0, 'ro', markersize=8, label='解 1: IRR = 10.0% (NPV=0)')
plt.plot(20, 0, 'go', markersize=8, label='解 2: IRR = 20.0% (NPV=0)')

plt.title("キャッシュフローの符号が2回反転するプロジェクトの NPV 曲線", fontsize=14)
plt.xlabel("割引率・年利 r (%)", fontsize=12)
plt.ylabel("正味現在価値 NPV (億円)", fontsize=12)
plt.legend(fontsize=11)
plt.show()


グラフを見ると明らかなように、**$\text{NPV}(r) = 0$ となる解（IRR）が「10%」と「20%」の2つ存在します！**

### ニュートン法を適用すると何が起きるか？
1. **初期値によって求まる解が変わる**:
   - $r_0 = 5\%$ から始めると $\rightarrow$ **$10\%$** に収束
   - $r_0 = 25\%$ から始めると $\rightarrow$ **$20\%$** に収束
2. **頂点付近（傾きがゼロ $f'(r) \approx 0$）を踏むと大爆発**:
   - 放物線の頂点（$r \approx 14.78\%$ 付近）では接線の傾きがほぼ水平になります。
   - この付近を初期値にすると、分母が極小になって**値が宇宙の彼方に吹き飛び、永久に収束しません**。

これがまさに、**Excel の `IRR` 関数が突然 `#NUM!` エラーを吐き出す最大の原因**です！


In [ ]:
# 頂点付近 (r = 14.78%) からニュートン法を開始してみる
r = 0.1478

print(f"危険な初期値: r_0 = {r * 100:.2f}% (傾きがほぼゼロの頂点付近)")
print("-" * 65)

for step in range(1, 8):
    val = npv_mine(r)
    slope = npv_mine_prime(r)
    r_next = r - (val / slope)
    print(f"Step {step}: 傾き={slope:.4e} --> 次の r = {r_next * 100:+.2f}%")
    r = r_next

print("-" * 65)
print("❌ 傾きがゼロに近いため、解が激しく振動・発散して求まりません（#NUM! エラー再現）")


---
## 6. まとめ：実務でのアルゴリズム選択

本講義で学んだ2つの手法（二分法とニュートン法）の長所と短所を整理してみましょう。

| 比較項目 | 二分法 (第2回) | ニュートン法 (第3回) |
| :--- | :--- | :--- |
| **計算スピード** | 1次収束（ゆっくり着実に進む） | **2次収束（猛烈に速い）** |
| **導関数の計算** | **不要（数式の形だけでOK）** | 必須（$f'(x)$ を計算する必要がある） |
| **収束の確実性** | **100% 確実に解に到達する** | 初期値次第で発散・振動・クラッシュする |
| **特異点の安全性** | **探索区間内しか調べないため安全** | 特異点（$r \le -1$）に飛び火する危険がある |

### 💡 現場・実務の知恵
1. **Excel の `IRR` 関数の工夫**:
   - 実は Excel の IRR 関数には `=IRR(範囲, [推定値])` という第2引数があります。
   - `#NUM!` エラーが出たときは、この第2引数に「おそらく10%くらい」と初期値（推定値）を教えてあげることで、ニュートン法を正しい解へ誘導できます。
2. **現代の標準「ブレント法（Brent法）」**:
   - 「二分法の絶対に失敗しない安全性」と「ニュートン法系の超高速性」を融合したハイブリッド手法です。
   - 基本は高速な補間法で進みつつ、解が飛びそうになったり収束が遅くなったら**自動的に二分法にブレーキを踏む**仕組みになっており、Pythonの `scipy` や金融工学ライブラリで標準採用されています。

> **結論**:  
> 「二分法は原始的で遅いから使えない」のではなく、**「絶対にシステムを落とせない（安全第一の）場面では、二分法こそが最も信頼できる最後の砦である」** というのが数値計算の極めて重要な教訓です。
